In [2]:
import numpy as np

In [3]:
import tensorflow as tf
# from tensorflow.keras import models, layers

c:\Users\BLESSED-PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.3.0)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


1. DATA PRE-PROCESSING

In [4]:
# train_ds = tf.keras.utils.image_dataset_from_directory(
#     'data/',
#     validation_split=0.2,
#     subset='training',
#     seed=123,
#     image_size=(224, 224),
#     batch_size=32
# )

In [5]:
# val_ds = tf.keras.utils.image_dataset_from_directory(
#     'data/',
#     validation_split=0.2,
#     subset='validation',
#     seed=123,
#     image_size=(224, 224),
#     batch_size=32
# )

2. MODEL LOADING

In [6]:
# base_model = tf.keras.applications.ResNet50(
#     weights='imagenet',
#     include_top=False,
#     input_shape=(224, 224, 3)
# )

# base_model.trainable = False

In [7]:
# base_model.save('resnet50_local.keras')

In [8]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess(image, label):
    image = tf.image.resize(image, (224, 224))
    image = preprocess_input(image)
    return image, label

# train_ds = train_ds.map(preprocess)
# val_ds = val_ds.map(preprocess)

In [9]:
# model = models.Sequential([
#     layers.Input(shape=(224, 224, 3)),
#     base_model,
#     layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.5),
#     layers.Dense(1, activation='sigmoid')
# ])

In [10]:
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [11]:
# model.fit(train_ds, validation_data=val_ds, epochs=5)

In [12]:
# model.save('face_mask_model.keras')

TEST

LOAD MODEL

In [13]:
model = tf.keras.models.load_model("face_mask_model.keras")

VIDEO TEST

In [14]:
import cv2

In [15]:
def detect_face_mask(img):
    image = cv2.resize(img, (224, 224))
    image = preprocess_input(image)
    y_pred = model.predict(image.reshape(1, 224, 224, 3))
    return round(y_pred[0][0])

0 - MASK ON
1 - MASK OFF

In [16]:
sample1 = cv2.imread('data/without_mask/without_mask_4.jpg')
sample2 = cv2.imread('data/with_mask/with_mask_7.jpg')

In [17]:
detect_face_mask(sample1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


1

In [18]:
detect_face_mask(sample2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step


0

In [48]:
def draw_label(img, text, pos, bg_color):
    text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, cv2.FILLED)

    end_x = pos[0] + text_size[0][0] + 1
    end_y = pos[1] + text_size[0][1] - 1

    cv2.rectangle(img, pos, (end_x, end_y), bg_color, cv2.FILLED)
    cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 1, cv2.LINE_AA)

NEED THE detect_face_mask function to work only on face co-ordinates

In [49]:
#GET Face co-ordinates
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

In [52]:
def detect_face(img):
    coords = face_cascade.detectMultiScale(img, scaleFactor=1.1, minNeighbors=10, minSize=(30,30))
    return coords

In [55]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    coords = detect_face(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))

    for x,y,w,h in coords:
        cv2.rectangle(frame, (x,y), (x+w,y+h), (255,0,0), 2)
        face = frame[y:y+h, x:x+w]
        y_pred = detect_face_mask(face)
        print(y_pred)

    if y_pred == 0:
        draw_label(frame, 'MASK ON', (30,30), (0, 255, 0))
    else:
        draw_label(frame, 'MASK OFF', (30,30), (0, 0, 255))     

    cv2.imshow('window', frame)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break
cap.release()
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
1
